# LeetCode 979 — Distribute Coins in Binary Tree bằng Python

Link đề: https://leetcode.com/problems/distribute-coins-in-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree (cây), DFS (duyệt theo chiều sâu), Binary Tree (cây nhị phân)

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def distributeCoins(self, root: Optional[TreeNode]) -> int:
        ...
```

Ghi chú: mỗi nút của cây nhị phân có `node.val` đồng xu và **tổng số xu đúng bằng số nút**. Mỗi lượt ta chuyển **một** đồng xu giữa hai nút **kề nhau** (cha ↔ con). Cần **số lượt ít nhất** để mọi nút có **đúng 1** đồng xu. Ý tưởng then chốt: với mỗi **cạnh** cha-con, số xu phải đi qua cạnh đó (theo cả hai chiều) bằng **|số xu trong cây con − số nút trong cây con|**; đáp án là tổng các giá trị này trên mọi cạnh. Bài này **tự dựng `TreeNode`** và hàm dựng cây theo danh sách kiểu LeetCode để kiểm thử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuyển xu qua cạnh cây, mục tiêu "mỗi nút đúng một xu", đếm số lượt ít nhất.
2. Nắm các công cụ Python cần thiết: lớp `TreeNode`, hàm dựng cây từ danh sách theo mức (có `None`), đệ quy hậu tự (post-order) trả về giá trị, biến `nonlocal`, `collections.deque` cho BFS, và từ điển/danh sách phụ để lưu cha của từng nút.
3. Hiểu nguyên lý **"thặng dư" (surplus) của cây con** `= số xu − số nút`: nếu dương thì dư và phải đẩy ra, nếu âm thì thiếu và phải nhận vào; cạnh nối cây con với cha chịu đúng `|thặng dư|` lượt chuyển.
4. Đi qua 4 cách: BFS trên mọi phân bố xu (oracle theo đúng định nghĩa, chỉ cho `n` nhỏ), tính `|thặng dư|` từng cạnh bằng cách duyệt lại từng cây con, DFS hậu tự trả về thặng dư (cách nộp chính), và duyệt BFS rồi xử lý **ngược thứ tự** với mảng cha (không đệ quy).
5. Tránh các lỗi kinh điển: trả về **giá trị tuyệt đối** thay vì thặng dư có dấu, **quên `abs`** nên các số âm dương triệt tiêu nhau, **quên trừ 1** cho chính nút đó, chỉ đếm cạnh dương, và chỉ đếm số xu thừa mà bỏ qua khoảng cách.

## 2. Đề bài & ví dụ

Cho `root` của cây nhị phân có `n` nút, trong đó mỗi nút có `node.val` đồng xu. Tổng số xu trên cây đúng bằng `n`.

Trong **một lượt**, ta chọn hai nút **kề nhau** và chuyển **một** đồng xu từ nút này sang nút kia (cha sang con hoặc con sang cha).

Trả về **số lượt ít nhất** để mỗi nút có **đúng một** đồng xu.

**Ví dụ 1:** `root = [3,0,0]` -> `2`. Từ gốc chuyển một xu sang con trái và một xu sang con phải.

**Ví dụ 2:** `root = [0,3,0]` -> `3`. Từ con trái chuyển hai xu lên gốc (2 lượt), rồi từ gốc chuyển một xu sang con phải (1 lượt).

**Ràng buộc:**

- Số nút trong cây là `n`, `1 <= n <= 100`.
- `0 <= Node.val <= n`.
- Tổng mọi `Node.val` bằng `n`.

In [1]:
from typing import List, Optional, Dict, Tuple
from collections import deque
import heapq
import random, time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(vals: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách theo mức kiểu LeetCode (None là nút trống; chỉ nút thật có hai phần tử con trong danh sách)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = deque([root]), 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

def tree_to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Xuất cây theo mức kiểu LeetCode (bỏ None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def count_nodes(root: Optional[TreeNode]) -> int:
    return 0 if root is None else 1 + count_nodes(root.left) + count_nodes(root.right)

def total_coins(root: Optional[TreeNode]) -> int:
    return 0 if root is None else root.val + total_coins(root.left) + total_coins(root.right)

examples = [
    ([3, 0, 0], 2),
    ([0, 3, 0], 3),
]
for vals, expected in examples:
    t = build_tree(vals)
    print(f"root={vals} -> {count_nodes(t)} nút, {total_coins(t)} xu, danh sách theo mức {tree_to_list(t)}, kỳ vọng {expected}")
    assert tree_to_list(t) == vals and total_coins(t) == count_nodes(t)

root=[3, 0, 0] -> 3 nút, 3 xu, danh sách theo mức [3, 0, 0], kỳ vọng 2
root=[0, 3, 0] -> 3 nút, 3 xu, danh sách theo mức [0, 3, 0], kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Cây nhị phân, `TreeNode` và danh sách theo mức

LeetCode cho cây dưới dạng danh sách theo **từng mức**, trong đó `None` là nút trống và **nút trống không có con trong danh sách**. `build_tree` dựng cây từ danh sách bằng hàng đợi; `tree_to_list` làm ngược lại. Ta dùng chúng để kiểm thử (LeetCode tự dựng cây trước khi gọi hàm của bạn).

In [2]:
t = build_tree([0, None, 2, None, None])        # None thừa ở cuối vẫn được
print(tree_to_list(t), "| gốc:", t.val, "| con phải:", t.right.val, "| con trái:", t.left)
print(tree_to_list(build_tree([1, 2, None, 3])))

[0, None, 2] | gốc: 0 | con phải: 2 | con trái: None
[1, 2, None, 3]


### 3.2. Đệ quy hậu tự (post-order)

Hậu tự xử lý **hai cây con trước, rồi mới đến nút hiện tại**: lý tưởng khi giá trị của nút cha phụ thuộc kết quả của các con. Hàm `dfs(node)` trả về một số cho cây con của `node`; với `None` trả về `0`.

In [3]:
def subtree_size(node) -> int:
    if node is None:
        return 0
    left = subtree_size(node.left)       # xử lý con trái trước
    right = subtree_size(node.right)     # rồi con phải
    return left + right + 1              # cuối cùng mới tới nút hiện tại

print("kích thước cây [1,2,None,3]:", subtree_size(build_tree([1, 2, None, 3])))

kích thước cây [1,2,None,3]: 3


### 3.3. Thặng dư của cây con

Với cây con gốc `u`, đặt `surplus(u) = (tổng xu trong cây con) - (số nút trong cây con)`. Khi xong, cây con đó cần **đúng bằng** số nút, nên nó phải **gửi ra** `surplus(u)` xu nếu dương, hoặc **nhận vào** `-surplus(u)` xu nếu âm. Mọi xu đi vào/ra cây con **bắt buộc** đi qua **một cạnh duy nhất** nối `u` với cha của `u`.

In [4]:
def coins_and_size(node) -> Tuple[int, int]:
    if node is None:
        return 0, 0
    lc, ls = coins_and_size(node.left)
    rc, rs = coins_and_size(node.right)
    return node.val + lc + rc, 1 + ls + rs

t = build_tree([0, 3, 0])
for name, sub in (("cây con trái", t.left), ("cây con phải", t.right), ("toàn cây", t)):
    c, s = coins_and_size(sub)
    print(f"{name:13}: xu={c}, nút={s}, thặng dư={c - s}")

cây con trái : xu=3, nút=1, thặng dư=2
cây con phải : xu=0, nút=1, thặng dư=-1
toàn cây     : xu=3, nút=3, thặng dư=0


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `[0, 3, 0]` (gốc `0`, con trái `3`, con phải `0`).

- Cây con trái (chỉ một nút): 3 xu, 1 nút: thặng dư `+2`. Cạnh gốc-con trái phải chuyển `2` xu **lên** gốc: 2 lượt.
- Cây con phải: 0 xu, 1 nút: thặng dư `-1`. Cạnh gốc-con phải phải chuyển `1` xu **xuống**: 1 lượt.
- Toàn cây: 3 xu, 3 nút: thặng dư `0` (đúng như đề bảo đảm).

Tổng: `2 + 1 = 3` lượt.

Ví dụ 1: `[3, 0, 0]`: hai con đều có thặng dư `-1`: `1 + 1 = 2` lượt.

**Vì sao không thể ít hơn?** Mỗi lượt chỉ chuyển **một** xu qua **một** cạnh, mà cạnh `e` phải chịu ít nhất `|surplus|` xu đi qua. Cộng các cạnh ta có cận dưới; và ta sẽ chứng minh (mục 10) rằng cận này **đạt được**.

## 5. Cách 1 — Brute Force: BFS trên mọi phân bố xu

**Ý tưởng.** Một **trạng thái** là bộ số xu của các nút `(c0, c1, ..., c(n-1))`. Một **lượt** là chọn cạnh `(a, b)` với `ca >= 1` rồi chuyển một xu từ `a` sang `b`. Dùng **BFS** từ trạng thái ban đầu tới trạng thái `(1, 1, ..., 1)`: độ sâu tìm được chính là số lượt ít nhất, **theo đúng định nghĩa** (không dùng thặng dư). Số trạng thái là số cách chia `n` xu cho `n` nút, `C(2n - 1, n)`: `n = 8` là `6435`, nhưng `n = 100` là khổng lồ, nên chỉ dùng làm **oracle cho cây nhỏ**.

Ta tạo hàm gán chỉ số cho các nút theo BFS và danh sách cha để dùng chung ở các cách sau.

In [5]:
def index_tree(root: Optional[TreeNode]) -> Tuple[List[TreeNode], List[int]]:
    """Trả về (danh sách nút theo thứ tự BFS, danh sách chỉ số cha; gốc có cha = -1)."""
    nodes, parent = [root], [-1]
    i = 0
    while i < len(nodes):
        nd = nodes[i]
        for ch in (nd.left, nd.right):
            if ch is not None:
                nodes.append(ch)
                parent.append(i)
        i += 1
    return nodes, parent

def moves_bfs(root: Optional[TreeNode]) -> int:
    nodes, parent = index_tree(root)
    n = len(nodes)
    adj = [[] for _ in range(n)]
    for v in range(1, n):
        adj[v].append(parent[v])
        adj[parent[v]].append(v)
    start = tuple(nd.val for nd in nodes)
    goal = (1,) * n
    if start == goal:
        return 0
    seen = {start}
    dq = deque([(start, 0)])
    while dq:
        cur, d = dq.popleft()
        for a in range(n):
            if cur[a] == 0:
                continue
            for b in adj[a]:
                nxt = list(cur)
                nxt[a] -= 1
                nxt[b] += 1
                nxt = tuple(nxt)
                if nxt in seen:
                    continue
                if nxt == goal:
                    return d + 1
                seen.add(nxt)
                dq.append((nxt, d + 1))
    raise AssertionError("tổng xu = số nút nên luôn đến được trạng thái đích")

for vals, expected in examples:
    got = moves_bfs(build_tree(vals))
    print(f"root={vals}: BFS = {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[3, 0, 0]: BFS = 2 OK
root=[0, 3, 0]: BFS = 3 OK


**Độ phức tạp và nút thắt.** Số trạng thái `C(2n - 1, n)` tăng gần như `4^n`, mỗi trạng thái có `O(n)` lượt: chỉ dùng được tới `n ~ 9`. Đo thử:

In [6]:
def random_tree(rng: random.Random, n: int, coins: bool = True) -> TreeNode:
    """Cây ngẫu nhiên n nút (gắn nút mới vào một ô trống ngẫu nhiên) với n xu chia ngẫu nhiên."""
    root = TreeNode(0)
    slots = [(root, "left"), (root, "right")]
    nodes = [root]
    for _ in range(n - 1):
        parent, side = slots.pop(rng.randrange(len(slots)))
        child = TreeNode(0)
        setattr(parent, side, child)
        nodes.append(child)
        slots += [(child, "left"), (child, "right")]
    if coins:
        for _ in range(n):
            rng.choice(nodes).val += 1
    return root

rng = random.Random(979)
for n in (4, 6, 8, 9):
    t = random_tree(rng, n)
    t0 = time.perf_counter()
    ans = moves_bfs(t)
    print(f"n={n}: {1000 * (time.perf_counter() - t0):9.1f} ms, đáp án {ans}")
print("n = 100: số trạng thái C(199, 100) ~ 10^58 -> không thể")

n=4:       0.0 ms, đáp án 2
n=6:       0.4 ms, đáp án 7
n=8:       8.4 ms, đáp án 13
n=9:       3.8 ms, đáp án 7
n = 100: số trạng thái C(199, 100) ~ 10^58 -> không thể


Nút thắt: ta không cần duyệt từng lượt; chỉ cần biết **mỗi cạnh phải chịu bao nhiêu xu**, và con số đó có công thức đóng là `|thặng dư|` của cây con.

## 6. Ý tưởng thuật toán / bước đột phá

**Đáp án = tổng `|thặng dư|` của mọi cây con ứng với một cạnh (tức mọi nút không phải gốc).**

- Cận dưới: cây con của `u` cần `surplus(u)` xu đi ra (hoặc vào) qua cạnh `(u, cha)`, và mỗi lượt chỉ chuyển một xu qua một cạnh, nên cạnh đó tốn **ít nhất** `|surplus(u)|` lượt.
- Cận trên: tồn tại một kế hoạch dùng **đúng** `|surplus|` lượt trên mỗi cạnh (mục 10): trước hết đẩy mọi xu dư **lên** theo hậu tự, sau đó phát xu xuống nơi thiếu theo tiền tự.

Vậy mỗi cạnh được tính một lần với `|surplus|`, và ta chỉ cần **một lượt duyệt** để tính mọi thặng dư.

| Cách | Ý tưởng |
|---|---|
| 2 | Với mỗi nút, duyệt lại cây con để đếm xu và nút: `O(n^2)` |
| 3 | DFS hậu tự trả về thặng dư của cây con, cộng `|thặng dư|` của hai con (nộp): `O(n)` |
| 4 | Không đệ quy: duyệt BFS, rồi xử lý ngược để dồn thặng dư lên cha |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tính thặng dư từng cạnh bằng cách duyệt lại cây con

**Giải thích từng đoạn.**

- Lấy danh sách nút theo BFS (`index_tree`).
- Với mỗi nút `u` khác gốc, tính `coins_and_size(u)` (duyệt toàn bộ cây con của `u`), lấy `|coins - size|`.
- Cộng lại. Mỗi nút bị duyệt lại nhiều lần (một lần cho mỗi tổ tiên): `O(n * độ sâu)`, tồi nhất `O(n^2)` (cây dạng dây).

In [7]:
def moves_edges(root: Optional[TreeNode]) -> int:
    nodes, parent = index_tree(root)
    total = 0
    for v in range(1, len(nodes)):                 # mỗi nút khác gốc <-> một cạnh (v, cha của v)
        coins, size = coins_and_size(nodes[v])
        total += abs(coins - size)
    return total

for vals, expected in examples:
    got = moves_edges(build_tree(vals))
    print(f"root={vals}: tổng |thặng dư| các cạnh = {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[3, 0, 0]: tổng |thặng dư| các cạnh = 2 OK
root=[0, 3, 0]: tổng |thặng dư| các cạnh = 3 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def distributeCoins(self, root: Optional[TreeNode]) -> int:
        def coins_and_size(node):
            if node is None:
                return 0, 0
            lc, ls = coins_and_size(node.left)
            rc, rs = coins_and_size(node.right)
            return node.val + lc + rc, 1 + ls + rs

        total = 0
        stack = [root.left, root.right]               # mọi nút khác gốc, duyệt bằng ngăn xếp
        while stack:
            node = stack.pop()
            if node is None:
                continue
            coins, size = coins_and_size(node)
            total += abs(coins - size)
            stack.append(node.left)
            stack.append(node.right)
        return total

sol_edges = Solution().distributeCoins
for vals, expected in examples:
    assert sol_edges(build_tree(vals)) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n^2)` trong tồi nhất (cây dạng dây: tổng kích thước các cây con `~ n^2 / 2`); bộ nhớ `O(n)` cho ngăn xếp/đệ quy. Với `n <= 100` vẫn rất nhanh, nhưng lãng phí vì ta tính lại các tổng đã có.

---

### 7.2. Cách 3 — DFS hậu tự trả về thặng dư (cách nộp chính)

**Ý tưởng.** Thay vì duyệt lại, để `dfs(node)` **trả về thặng dư có dấu** của cây con gốc `node`:

`dfs(node) = node.val - 1 + dfs(node.left) + dfs(node.right)` (và `dfs(None) = 0`).

Cạnh từ `node` xuống con trái chịu `|dfs(node.left)|` lượt, cạnh xuống con phải chịu `|dfs(node.right)|` lượt. Ta cộng hai giá trị này vào `moves` ngay khi hai con vừa được tính.

**Giải thích từng đoạn.**

- `dfs(None) = 0`: cây rỗng không có xu và không có nút.
- `left, right = dfs(node.left), dfs(node.right)`: thặng dư có dấu của hai cây con.
- `moves += abs(left) + abs(right)`: mỗi cạnh được tính **đúng một lần** ở nút cha của nó.
- Trả về `node.val - 1 + left + right` (**giữ dấu**, không phải giá trị tuyệt đối).

In [9]:
def moves_dfs(root: Optional[TreeNode]) -> int:
    moves = 0

    def dfs(node: Optional[TreeNode]) -> int:
        nonlocal moves
        if node is None:
            return 0
        left, right = dfs(node.left), dfs(node.right)
        moves += abs(left) + abs(right)
        return node.val - 1 + left + right

    dfs(root)
    return moves

for vals, expected in examples:
    got = moves_dfs(build_tree(vals))
    print(f"root={vals}: DFS = {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[3, 0, 0]: DFS = 2 OK
root=[0, 3, 0]: DFS = 3 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def distributeCoins(self, root: Optional[TreeNode]) -> int:
        moves = 0

        def dfs(node: Optional[TreeNode]) -> int:
            nonlocal moves
            if node is None:
                return 0
            left, right = dfs(node.left), dfs(node.right)
            moves += abs(left) + abs(right)          # mỗi cạnh tới con chịu |thặng dư| của cây con
            return node.val - 1 + left + right      # thặng dư CÓ DẤU của cây con gốc node

        dfs(root)
        return moves

sol_dfs = Solution().distributeCoins
for vals, expected in examples:
    assert sol_dfs(build_tree(vals)) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n)` (mỗi nút thăm một lần); bộ nhớ `O(h)` cho ngăn xếp đệ quy, `h` là chiều cao cây (`h <= n <= 100`, an toàn so với giới hạn đệ quy mặc định `1000`).

---

### 7.3. Cách 4 — BFS rồi xử lý ngược thứ tự (không đệ quy)

**Ý tưởng khác.** Thứ tự BFS luôn liệt kê **cha trước con**. Nếu duyệt danh sách đó **từ cuối về đầu** thì mỗi nút được xử lý **sau tất cả hậu duệ** của nó, giống hậu tự. Ta giữ mảng `excess[v]` = số xu nút `v` đang "giữ" trừ 1; khi xử lý `v`, cộng `|excess[v]|` vào đáp án rồi **chuyển `excess[v]` lên cha**.

**Giải thích từng đoạn.**

- `nodes, parent = index_tree(root)`: danh sách nút theo BFS và chỉ số cha.
- `excess = [node.val - 1 ...]`.
- Với `v` từ `n - 1` về `1`: `ans += abs(excess[v])`; `excess[parent[v]] += excess[v]`.
- Gốc (chỉ số `0`) không có cạnh lên cha nên không cộng gì.

In [11]:
def moves_reverse_bfs(root: Optional[TreeNode]) -> int:
    nodes, parent = index_tree(root)
    excess = [nd.val - 1 for nd in nodes]
    ans = 0
    for v in range(len(nodes) - 1, 0, -1):
        ans += abs(excess[v])
        excess[parent[v]] += excess[v]
    return ans

for vals, expected in examples:
    got = moves_reverse_bfs(build_tree(vals))
    print(f"root={vals}: BFS ngược = {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[3, 0, 0]: BFS ngược = 2 OK
root=[0, 3, 0]: BFS ngược = 3 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def distributeCoins(self, root: Optional[TreeNode]) -> int:
        from collections import deque
        nodes, parent = [root], [-1]
        i = 0
        while i < len(nodes):                         # BFS: cha luôn đứng trước con
            nd = nodes[i]
            for ch in (nd.left, nd.right):
                if ch is not None:
                    nodes.append(ch)
                    parent.append(i)
            i += 1
        excess = [nd.val - 1 for nd in nodes]
        ans = 0
        for v in range(len(nodes) - 1, 0, -1):        # con trước cha
            ans += abs(excess[v])
            excess[parent[v]] += excess[v]
        return ans

sol_rbfs = Solution().distributeCoins
for vals, expected in examples:
    assert sol_rbfs(build_tree(vals)) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n)`; bộ nhớ `O(n)` cho hai danh sách (không dùng ngăn xếp đệ quy, hữu ích khi cây rất sâu).

## 8. TRACE từng bước

### 8.1. DFS cho `[0, 3, 0]`

In từng lần trả về của `dfs` (thụt lề theo độ sâu).

In [13]:
def trace_dfs(root):
    moves = 0

    def dfs(node, depth):
        nonlocal moves
        pad = "  " * depth
        if node is None:
            print(f"{pad}dfs(None) = 0")
            return 0
        print(f"{pad}vào nút val={node.val}")
        left = dfs(node.left, depth + 1)
        right = dfs(node.right, depth + 1)
        moves += abs(left) + abs(right)
        res = node.val - 1 + left + right
        print(f"{pad}nút val={node.val}: left={left}, right={right}, moves += {abs(left) + abs(right)} (moves={moves}) -> trả về {res}")
        return res

    dfs(root, 0)
    return moves

got = trace_dfs(build_tree([0, 3, 0]))
print("đáp án:", got)
assert got == 3

vào nút val=0
  vào nút val=3
    dfs(None) = 0
    dfs(None) = 0
  nút val=3: left=0, right=0, moves += 0 (moves=0) -> trả về 2
  vào nút val=0
    dfs(None) = 0
    dfs(None) = 0
  nút val=0: left=0, right=0, moves += 0 (moves=0) -> trả về -1
nút val=0: left=2, right=-1, moves += 3 (moves=3) -> trả về 0
đáp án: 3


### 8.2. Cách 4 cho `[0, 3, 0]`: xử lý ngược thứ tự BFS

`nodes` theo BFS: chỉ số `0` = gốc (`0` xu), `1` = con trái (`3`), `2` = con phải (`0`). `excess = [-1, 2, -1]`. Xử lý `v = 2`: cộng `1`, chuyển `-1` lên gốc. Xử lý `v = 1`: cộng `2`, chuyển `+2` lên gốc. Tổng `3`.

In [14]:
def trace_rbfs(root):
    nodes, parent = index_tree(root)
    excess = [nd.val - 1 for nd in nodes]
    print("nodes (val):", [nd.val for nd in nodes], "| parent:", parent, "| excess:", excess)
    ans = 0
    for v in range(len(nodes) - 1, 0, -1):
        ans += abs(excess[v])
        excess[parent[v]] += excess[v]
        print(f"v={v}: ans += {abs(excess[v])} -> {ans}; excess[{parent[v]}] = {excess[parent[v]]}")
    print("excess cuối của gốc (phải bằng 0):", excess[0])
    assert excess[0] == 0
    return ans

assert trace_rbfs(build_tree([0, 3, 0])) == 3

nodes (val): [0, 3, 0] | parent: [-1, 0, 0] | excess: [-1, 2, -1]
v=2: ans += 1 -> 1; excess[0] = -2
v=1: ans += 2 -> 3; excess[0] = 0
excess cuối của gốc (phải bằng 0): 0


### 8.3. Thứ tự chuyển xu thực tế cho `[0, 3, 0]`

Đáp án `3` đi kèm một **kế hoạch cụ thể**: hai lượt `con trái -> gốc` rồi một lượt `gốc -> con phải`. Ta sẽ tự dựng kế hoạch này và mô phỏng tổng quát ở mục 10.

## 9. Code nộp LeetCode

Ta chọn **Cách 3** (DFS hậu tự): ngắn, `O(n)`, và dễ chứng minh đúng.

```python
class Solution:
    def distributeCoins(self, root: Optional[TreeNode]) -> int:
        moves = 0

        def dfs(node: Optional[TreeNode]) -> int:
            nonlocal moves
            if node is None:
                return 0
            left, right = dfs(node.left), dfs(node.right)
            moves += abs(left) + abs(right)          # mỗi cạnh tới con chịu |thặng dư| của cây con
            return node.val - 1 + left + right      # thặng dư CÓ DẤU của cây con gốc node

        dfs(root)
        return moves
```

In [15]:
class Solution:
    def distributeCoins(self, root: Optional[TreeNode]) -> int:
        moves = 0

        def dfs(node: Optional[TreeNode]) -> int:
            nonlocal moves
            if node is None:
                return 0
            left, right = dfs(node.left), dfs(node.right)
            moves += abs(left) + abs(right)          # mỗi cạnh tới con chịu |thặng dư| của cây con
            return node.val - 1 + left + right      # thặng dư CÓ DẤU của cây con gốc node

        dfs(root)
        return moves

sol = Solution()
for vals, expected in examples:
    got = sol.distributeCoins(build_tree(vals))
    print(f"root={vals}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[3, 0, 0]: 2 OK
root=[0, 3, 0]: 3 OK


## 10. Vì sao đúng

**(a) Cận dưới.** Với nút `u` khác gốc, cây con `T(u)` có `coins(u)` xu và `size(u)` nút. Lúc kết thúc `T(u)` có đúng `size(u)` xu. Xu chỉ vào/ra `T(u)` qua **một** cạnh `(u, cha(u))`, mỗi lượt đổi số xu của `T(u)` đúng `1`. Vậy cạnh này được dùng **ít nhất** `|coins(u) - size(u)|` lượt. Các cạnh khác nhau là các lượt khác nhau, nên tổng số lượt `>= Σ |surplus(u)|`.

**(b) Cận trên (xây kế hoạch).** Gọi `s(u) = surplus(u)`. Kế hoạch gồm hai pha:

1. **Pha lên (hậu tự):** với mỗi `u` có `s(u) > 0`, chuyển `s(u)` xu từ `u` lên `cha(u)`. Lúc đó `u` đang giữ `val(u) + Σ_{con có s > 0} s(con)` xu, và `giữ - s(u) = 1 + Σ_{con có s < 0} |s(con)| >= 1`, nên **đủ xu** để chuyển.
2. **Pha xuống (tiền tự):** với mỗi `u` có `s(u) < 0`, chuyển `|s(u)|` xu từ `cha(u)` xuống `u`. Khi tới lượt phát cho các con của `p`, nút `p` đang giữ `1 + Σ_{con có s < 0} |s(con)|` xu (tính theo công thức của `s(p)`), vừa **đủ** để phát cho các con thiếu và giữ lại `1`.

Mỗi cạnh dùng đúng `|s(u)|` lượt: tổng đúng `Σ |s(u)|`. Cận trên khớp cận dưới nên đáp án là `Σ |s(u)|`.

**(c) Thuật toán.** `dfs(node)` trả về `s(node)` bằng quy nạp: `s(node) = val(node) - 1 + s(left) + s(right)` (vì `coins` và `size` đều cộng dồn). Cộng `|s(left)| + |s(right)|` tại mỗi nút `node` tính mỗi cạnh đúng một lần. Gốc không có cạnh lên nên không cộng `|s(root)|` (và nó bằng `0` vì tổng xu bằng số nút).

**Kiểm chứng bằng máy.** Ta (1) tự dựng kế hoạch hai pha rồi **mô phỏng** để chắc rằng không bao giờ cần chuyển xu từ nút hết xu, kết quả cuối mọi nút đúng một xu, và số lượt bằng công thức; (2) so với **BFS theo định nghĩa**: với mọi hình dạng cây nhỏ ta tính bảng khoảng cách bằng một lần BFS **từ trạng thái đích** (mỗi lượt đảo ngược được: chuyển xu `a -> b` rồi `b -> a`, nên khoảng cách đối xứng) và đối chiếu **mọi** cách phân bố xu có thể.

In [16]:
def plan_moves(root: Optional[TreeNode]) -> List[Tuple[int, int]]:
    """Trả về danh sách lượt (từ nút a sang nút b) theo kế hoạch hai pha; nút đánh chỉ số theo BFS."""
    nodes, parent = index_tree(root)
    n = len(nodes)
    s = [nd.val - 1 for nd in nodes]
    for v in range(n - 1, 0, -1):
        s[parent[v]] += s[v]
    moves = []
    for v in range(n - 1, 0, -1):                        # pha lên: con trước cha (hậu tự)
        if s[v] > 0:
            moves += [(v, parent[v])] * s[v]
    for v in range(1, n):                                # pha xuống: cha trước con (tiền tự)
        if s[v] < 0:
            moves += [(parent[v], v)] * (-s[v])
    return moves

def simulate(root: Optional[TreeNode], moves: List[Tuple[int, int]]) -> List[int]:
    nodes, parent = index_tree(root)
    coins = [nd.val for nd in nodes]
    for a, b in moves:
        assert parent[a] == b or parent[b] == a, "chỉ chuyển giữa hai nút kề nhau"
        assert coins[a] >= 1, "nút hết xu mà vẫn chuyển"
        coins[a] -= 1
        coins[b] += 1
    return coins

for vals, expected in examples:
    t = build_tree(vals)
    mv = plan_moves(t)
    final = simulate(t, mv)
    print(f"root={vals}: kế hoạch {mv} -> xu cuối {final}")
    assert final == [1] * len(final) and len(mv) == expected

rng = random.Random(1979)
for _ in range(500):
    n = rng.randint(1, 100)
    t = random_tree(rng, n)
    mv = plan_moves(t)
    final = simulate(t, mv)
    assert final == [1] * n and len(mv) == moves_dfs(t)
print("kế hoạch hai pha hợp lệ trên 500 cây ngẫu nhiên (n tới 100): số lượt = công thức")

root=[3, 0, 0]: kế hoạch [(0, 1), (0, 2)] -> xu cuối [1, 1, 1]
root=[0, 3, 0]: kế hoạch [(1, 0), (1, 0), (0, 2)] -> xu cuối [1, 1, 1]
kế hoạch hai pha hợp lệ trên 500 cây ngẫu nhiên (n tới 100): số lượt = công thức


In [17]:
def shape_table(parent: List[int]) -> Dict[tuple, int]:
    """BFS từ trạng thái đích: khoảng cách (đối xứng) từ mọi phân bố xu tới (1, ..., 1)."""
    n = len(parent)
    adj = [[] for _ in range(n)]
    for v in range(1, n):
        adj[v].append(parent[v])
        adj[parent[v]].append(v)
    goal = (1,) * n
    dist = {goal: 0}
    dq = deque([goal])
    while dq:
        cur = dq.popleft()
        d = dist[cur]
        for a in range(n):
            if cur[a] == 0:
                continue
            for b in adj[a]:
                nxt = list(cur)
                nxt[a] -= 1
                nxt[b] += 1
                nxt = tuple(nxt)
                if nxt not in dist:
                    dist[nxt] = d + 1
                    dq.append(nxt)
    return dist

from math import comb
rng = random.Random(2979)
checked = 0
for n in range(1, 8):
    for _ in range(25 if n >= 4 else 4):
        shape = random_tree(rng, n, coins=False)
        nodes, parent = index_tree(shape)
        table = shape_table(parent)
        assert len(table) == comb(2 * n - 1, n)               # mọi phân bố xu đều đến được đích
        for state, d in table.items():
            for nd, c in zip(nodes, state):
                nd.val = c
            assert sol.distributeCoins(shape) == d, (parent, state, d)
            checked += 1
print("so với BFS theo định nghĩa: đã kiểm", checked, "(hình dạng cây, phân bố xu) với n <= 7: khớp tuyệt đối")

so với BFS theo định nghĩa: đã kiểm 58531 (hình dạng cây, phân bố xu) với n <= 7: khớp tuyệt đối


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS trạng thái | `O(C(2n - 1, n) * n)` | `O(C(2n - 1, n))` |
| 2. Duyệt lại cây con | `O(n^2)` | `O(n)` |
| 3. DFS hậu tự | `O(n)` | `O(h)` |
| 4. BFS ngược | `O(n)` | `O(n)` |

Đo thời gian ở cỡ tối đa (`n = 100`) trên cây ngẫu nhiên và cây dạng dây (độ sâu `100`):

In [18]:
def chain_tree(n: int, left: bool = False) -> TreeNode:
    """Cây dạng dây n nút; toàn bộ n xu đặt ở gốc."""
    root = TreeNode(n)
    cur = root
    for _ in range(n - 1):
        nxt = TreeNode(0)
        if left:
            cur.left = nxt
        else:
            cur.right = nxt
        cur = nxt
    return root

rng = random.Random(11)
n = 100
cases = [("ngẫu nhiên", random_tree(rng, n)), ("dây (phải)", chain_tree(n)), ("dây (trái)", chain_tree(n, left=True))]
for label, t in cases:
    ref = moves_dfs(t)
    for name, fn in (("Cách 2", sol_edges), ("Cách 3", sol_dfs), ("Cách 4", sol_rbfs), ("nộp", sol.distributeCoins)):
        t0 = time.perf_counter()
        for _ in range(50):
            got = fn(t)
        dt = (time.perf_counter() - t0) / 50 * 1000
        assert got == ref, name
        print(f"{label:12} {name:7}: {dt:8.3f} ms, đáp án {got}")
assert sol.distributeCoins(chain_tree(100)) == 100 * 99 // 2
print("dây 100 nút, mọi xu ở gốc: n(n-1)/2 =", 100 * 99 // 2)

ngẫu nhiên   Cách 2 :    0.114 ms, đáp án 154
ngẫu nhiên   Cách 3 :    0.019 ms, đáp án 154
ngẫu nhiên   Cách 4 :    0.031 ms, đáp án 154
ngẫu nhiên   nộp    :    0.023 ms, đáp án 154
dây (phải)   Cách 2 :    0.517 ms, đáp án 4950
dây (phải)   Cách 3 :    0.013 ms, đáp án 4950
dây (phải)   Cách 4 :    0.025 ms, đáp án 4950
dây (phải)   nộp    :    0.013 ms, đáp án 4950
dây (trái)   Cách 2 :    0.510 ms, đáp án 4950
dây (trái)   Cách 3 :    0.014 ms, đáp án 4950
dây (trái)   Cách 4 :    0.022 ms, đáp án 4950
dây (trái)   nộp    :    0.015 ms, đáp án 4950
dây 100 nút, mọi xu ở gốc: n(n-1)/2 = 4950


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle BFS), đối chiếu mọi cách, cây ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [19]:
def all_solutions():
    return {"edges": moves_edges, "dfs": moves_dfs, "rbfs": moves_reverse_bfs,
            "sol_edges": sol_edges, "sol_dfs": sol_dfs, "sol_rbfs": sol_rbfs, "final": Solution().distributeCoins}

def check_all(vals, expected=None, brute=True):
    root = build_tree(vals)
    assert total_coins(root) == count_nodes(root), "dữ liệu sai ràng buộc"
    results = {name: fn(build_tree(vals)) for name, fn in all_solutions().items()}
    ref = results["dfs"]
    assert len(set(results.values())) == 1, results
    if brute and count_nodes(root) <= 8:
        assert moves_bfs(build_tree(vals)) == ref, (vals, moves_bfs(build_tree(vals)), ref)
    if expected is not None:
        assert ref == expected, (vals, ref, expected)
    return ref

# 12.1 ví dụ đề
for vals, expected in examples:
    check_all(vals, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], 0),                          # một nút đã có 1 xu
    ([1, 1, 1], 0),                    # mọi nút đã đủ
    ([2, 0], 1),                       # gốc 2 xu, con trái 0: 1 lượt xuống
    ([0, 2], 1),                       # gốc 0, con trái 2: 1 lượt lên
    ([0, None, 2], 1),                 # con phải 2 xu: 1 lượt lên
    ([0, 0, 3], 3),                    # con trái thiếu 1 (1 lượt), con phải dư 2 (2 lượt)
    ([4, 0, 0, 0], 4),                 # gốc 4; con trái có con trái; con trái cần 2 (qua 2 cạnh): 1 + 2 + 1 = 4
    ([0, 5, 0, 0, 0], 5),              # tính tay ở bài tập 2
    ([1, 0, 2], 2),                    # con trái thiếu 1, con phải dư 1: 1 + 1
    ([1, 0, 0, 3], 4),                 # lá sâu 3 xu: cạnh lên 2, rồi 1, cộng cạnh phải 1: 2 + 1 + 1
    ([0, 0, 0, 0, 5], 8),              # lá 5 xu: +4 (4 lượt), lá kia -1 (1), nút trái +2 (2), con phải gốc -1 (1)
]
for vals, expected in edge:
    root = build_tree(vals)
    assert total_coins(root) == count_nodes(root), vals
    check_all(vals, expected)
print("12.2 edge case: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed so với oracle BFS)
def wrong_return_abs(root):                   # trả về |thặng dư| thay vì thặng dư có dấu
    moves = 0
    def dfs(node):
        nonlocal moves
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        moves += abs(l) + abs(r)
        return abs(node.val - 1 + l + r)
    dfs(root)
    return moves

def wrong_no_abs(root):                       # quên abs: các thặng dư âm dương triệt tiêu
    moves = 0
    def dfs(node):
        nonlocal moves
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        moves += l + r
        return node.val - 1 + l + r
    dfs(root)
    return moves

def wrong_no_minus_one(root):                 # quên trừ 1 cho chính nút
    moves = 0
    def dfs(node):
        nonlocal moves
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        moves += abs(l) + abs(r)
        return node.val + l + r
    dfs(root)
    return moves

def wrong_positive_only(root):                # chỉ đếm cạnh có thặng dư dương
    moves = 0
    def dfs(node):
        nonlocal moves
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        moves += max(l, 0) + max(r, 0)
        return node.val - 1 + l + r
    dfs(root)
    return moves

def wrong_excess_only(root):                  # chỉ đếm số xu thừa (val - 1 nếu val > 1), bỏ qua khoảng cách
    nodes, _ = index_tree(root)
    return sum(nd.val - 1 for nd in nodes if nd.val > 1)

def find_counter(wrong, seed=9, tries=20000):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 7)
        t = random_tree(rng, n)
        vals = tree_to_list(t)
        if wrong(build_tree(vals)) != moves_bfs(build_tree(vals)):
            return vals
    return None

counters = {}
for name, fn in (("trả về |thặng dư| thay vì có dấu", wrong_return_abs), ("quên abs (triệt tiêu)", wrong_no_abs),
                 ("quên trừ 1 cho chính nút", wrong_no_minus_one), ("chỉ đếm cạnh dương", wrong_positive_only),
                 ("chỉ đếm số xu thừa", wrong_excess_only)):
    vals = find_counter(fn)
    assert vals is not None, name
    counters[name] = vals
    print(f"{name:34} root={vals}: sai -> {fn(build_tree(vals))}, đúng (BFS) -> {moves_bfs(build_tree(vals))}")
    check_all(vals)
print("12.3 phản ví dụ: OK")

trả về |thặng dư| thay vì có dấu   root=[1, 2, None, None, 0]: sai -> 3, đúng (BFS) -> 1
quên abs (triệt tiêu)              root=[2, None, 1, None, 0]: sai -> -2, đúng (BFS) -> 2
quên trừ 1 cho chính nút           root=[1, None, 0, 1, 1, 2]: sai -> 10, đúng (BFS) -> 2
chỉ đếm cạnh dương                 root=[2, None, 1, None, 0]: sai -> 0, đúng (BFS) -> 2
chỉ đếm số xu thừa                 root=[1, None, 0, 1, 1, 2]: sai -> 1, đúng (BFS) -> 2
12.3 phản ví dụ: OK


In [21]:
# 12.4 cây ngẫu nhiên có seed: n từ 1 đến 100, nhiều dạng; các cách đối chiếu nhau, n nhỏ đối chiếu BFS
rng = random.Random(4979)
for _ in range(1500):
    n = rng.randint(1, 100)
    t = random_tree(rng, n)
    vals = tree_to_list(t)
    assert tree_to_list(build_tree(vals)) == vals
    check_all(vals, brute=(n <= 8))
print("12.4 ngẫu nhiên (1500 cây, n tới 100): OK")

# 12.5 dạng đặc biệt: dây trái/phải (mọi xu ở gốc hoặc ở nút sâu nhất), cây đầy đủ, mọi xu ở một lá
for n in (1, 2, 3, 10, 50, 100):
    for left in (False, True):
        t = chain_tree(n, left)
        assert sol.distributeCoins(t) == n * (n - 1) // 2
        # xu dồn ở nút sâu nhất
        deepest = t
        while deepest.left or deepest.right:
            deepest = deepest.left or deepest.right
        total = t
        # đặt lại: gốc 0, nút sâu nhất n
        cur = t
        while cur is not None:
            cur.val = 0
            cur = cur.left or cur.right
        deepest.val = n
        assert sol.distributeCoins(t) == n * (n - 1) // 2          # đối xứng: cũng n(n-1)/2
print("12.5 cây dạng dây: OK (n(n-1)/2 theo cả hai hướng)")

# 12.6 tính chất: bất biến theo phản chiếu; thặng dư gốc bằng 0; gán đủ xu thì đáp án 0
def mirror(node):
    return None if node is None else TreeNode(node.val, mirror(node.right), mirror(node.left))

rng = random.Random(6979)
for _ in range(300):
    n = rng.randint(1, 60)
    t = random_tree(rng, n)
    assert sol.distributeCoins(t) == sol.distributeCoins(mirror(t))
    nodes, _ = index_tree(t)
    assert sum(nd.val - 1 for nd in nodes) == 0
    for nd in nodes:
        nd.val = 1
    assert sol.distributeCoins(t) == 0
print("12.6 tính chất (phản chiếu, tổng thặng dư 0, đủ xu thì 0): OK")

12.4 ngẫu nhiên (1500 cây, n tới 100): OK
12.5 cây dạng dây: OK (n(n-1)/2 theo cả hai hướng)
12.6 tính chất (phản chiếu, tổng thặng dư 0, đủ xu thì 0): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả về giá trị tuyệt đối của thặng dư

Viết `return abs(node.val - 1 + left + right)` làm mất dấu: nút cha không biết cây con bên dưới **dư** hay **thiếu**, nên cộng thặng dư của hai con sai.

In [22]:
vals = counters["trả về |thặng dư| thay vì có dấu"]
print("root =", vals, "| sai ->", wrong_return_abs(build_tree(vals)), "| đúng ->", Solution().distributeCoins(build_tree(vals)))
assert wrong_return_abs(build_tree(vals)) != Solution().distributeCoins(build_tree(vals))

root = [1, 2, None, None, 0] | sai -> 3 | đúng -> 1


Vì sao sai: thặng dư của cây cha là **tổng đại số** của thặng dư các con (cộng cả số dương lẫn âm), chỉ lấy `abs` khi **đếm lượt qua cạnh**. **Đúng:** trả về giá trị có dấu, dùng `abs` chỉ khi cộng vào `moves`.

### Lỗi 2 — Quên `abs` khi cộng vào đáp án

`moves += left + right` làm số dương và âm triệt tiêu nhau (chiều chuyển xu không quan trọng, chỉ **số lượt**).

In [23]:
vals = counters["quên abs (triệt tiêu)"]
print("root =", vals, "| sai ->", wrong_no_abs(build_tree(vals)), "| đúng ->", Solution().distributeCoins(build_tree(vals)))
assert wrong_no_abs(build_tree(vals)) != Solution().distributeCoins(build_tree(vals))

root = [2, None, 1, None, 0] | sai -> -2 | đúng -> 2


Vì sao sai: một cạnh phải chuyển 2 xu xuống và một cạnh khác phải chuyển 2 xu lên là **4 lượt**, không phải `0`. **Đúng:** `moves += abs(left) + abs(right)`.

### Lỗi 3 — Quên trừ 1 cho chính nút

Mỗi nút cần giữ lại **một** xu, nên thặng dư là `val - 1 + ...`. Dùng `val + ...` coi như nút không cần xu nào.

In [24]:
vals = counters["quên trừ 1 cho chính nút"]
print("root =", vals, "| sai ->", wrong_no_minus_one(build_tree(vals)), "| đúng ->", Solution().distributeCoins(build_tree(vals)))
assert wrong_no_minus_one(build_tree(vals)) != Solution().distributeCoins(build_tree(vals))

root = [1, None, 0, 1, 1, 2] | sai -> 10 | đúng -> 2


Vì sao sai: thặng dư `= xu - số nút` nên mỗi nút đóng góp `val - 1`. **Đúng:** `return node.val - 1 + left + right`.

### Lỗi 4 — Chỉ đếm các cạnh có thặng dư dương

Cạnh có thặng dư **âm** (cây con thiếu xu) cũng tốn lượt, vì xu phải đi **xuống**.

In [25]:
vals = counters["chỉ đếm cạnh dương"]
print("root =", vals, "| sai ->", wrong_positive_only(build_tree(vals)), "| đúng ->", Solution().distributeCoins(build_tree(vals)))
assert wrong_positive_only(build_tree(vals)) != Solution().distributeCoins(build_tree(vals))

root = [2, None, 1, None, 0] | sai -> 0 | đúng -> 2


Vì sao sai: ví dụ 1 (`[3, 0, 0]`) chỉ có cạnh âm và đáp án `2`, nhưng cách sai cho `0`. **Đúng:** đếm `|thặng dư|` mọi cạnh.

### Lỗi 5 — Chỉ đếm số xu thừa, bỏ qua khoảng cách

Mỗi xu thừa phải đi qua **nhiều** cạnh tới nơi cần, nên số lượt là tổng **quãng đường**, không phải số xu.

In [26]:
vals = counters["chỉ đếm số xu thừa"]
print("root =", vals, "| sai ->", wrong_excess_only(build_tree(vals)), "| đúng ->", Solution().distributeCoins(build_tree(vals)))
assert wrong_excess_only(build_tree(vals)) != Solution().distributeCoins(build_tree(vals))
print("ví dụ 2 [0,3,0]: xu thừa =", wrong_excess_only(build_tree([0, 3, 0])), "nhưng đáp án =", Solution().distributeCoins(build_tree([0, 3, 0])))

root = [1, None, 0, 1, 1, 2] | sai -> 1 | đúng -> 2
ví dụ 2 [0,3,0]: xu thừa = 2 nhưng đáp án = 3


Vì sao sai: ở `[0, 3, 0]` có 2 xu thừa nhưng một xu phải đi 2 cạnh. **Đúng:** cộng `|thặng dư|` theo cạnh (mỗi cạnh một lần).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chuyển tài nguyên giữa các nút kề nhau để cân bằng" trên **cây** ⇒ tính **luồng qua mỗi cạnh** `= |dư/thiếu của cây con|`; đáp án là tổng, không cần mô phỏng.
- Đệ quy hậu tự **trả về một đại lượng cộng dồn** (ở đây thặng dư có dấu) và **cập nhật đáp án toàn cục** (`nonlocal`) là khuôn mẫu rất phổ biến cho bài cây.
- Mỗi cạnh được tính ở **nút cha** (khi biết giá trị của con) để không đếm trùng.
- Không muốn đệ quy: duyệt BFS rồi xử lý **ngược**, dùng mảng cha.
- Cùng họ: Binary Tree Cameras (968), Sum of Distances in Tree (834), Binary Tree Maximum Path Sum (124), Diameter of Binary Tree (543).
- Test nên có: một nút, cây dạng dây (đáp án `n(n-1)/2`), mọi xu ở lá, tổng thặng dư triệt tiêu, cây đã cân bằng sẵn.

## 15. Biến thể

1. **Cạnh có chi phí:** chuyển một xu qua cạnh `(v, cha(v))` tốn `w[v]`; đáp án là `Σ w[v] * |s(v)|` (chứng minh giống hệt vì cận dưới và kế hoạch hai pha dùng mỗi cạnh đúng `|s|` lần).
2. **Cây tổng quát / đồ thị cây** (nhiều con): cùng công thức, duyệt mọi con.
3. **Đồ thị có chu trình:** bài toán trở thành luồng chi phí nhỏ nhất (min-cost flow).
4. Bài liên quan: 968, 834, 543.

Biến thể 1 kiểm bằng oracle Dijkstra trên không gian trạng thái (cây nhỏ):

In [27]:
def weighted_moves(root: Optional[TreeNode], w: List[int]) -> int:
    nodes, parent = index_tree(root)
    excess = [nd.val - 1 for nd in nodes]
    ans = 0
    for v in range(len(nodes) - 1, 0, -1):
        ans += w[v] * abs(excess[v])
        excess[parent[v]] += excess[v]
    return ans

def weighted_oracle(root: Optional[TreeNode], w: List[int]) -> int:
    nodes, parent = index_tree(root)
    n = len(nodes)
    adj = [[] for _ in range(n)]
    for v in range(1, n):
        adj[v].append((parent[v], w[v]))
        adj[parent[v]].append((v, w[v]))
    start = tuple(nd.val for nd in nodes)
    goal = (1,) * n
    dist = {start: 0}
    pq = [(0, start)]
    while pq:
        d, cur = heapq.heappop(pq)
        if cur == goal:
            return d
        if d > dist[cur]:
            continue
        for a in range(n):
            if cur[a] == 0:
                continue
            for b, cost in adj[a]:
                nxt = list(cur)
                nxt[a] -= 1
                nxt[b] += 1
                nxt = tuple(nxt)
                if nxt not in dist or d + cost < dist[nxt]:
                    dist[nxt] = d + cost
                    heapq.heappush(pq, (d + cost, nxt))
    raise AssertionError

rng = random.Random(15)
for _ in range(60):
    n = rng.randint(2, 6)
    t = random_tree(rng, n)
    w = [0] + [rng.randint(1, 5) for _ in range(n - 1)]
    assert weighted_moves(t, w) == weighted_oracle(t, w), tree_to_list(t)
print("biến thể có chi phí cạnh: khớp Dijkstra trên 60 cây nhỏ")

biến thể có chi phí cạnh: khớp Dijkstra trên 60 cây nhỏ


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS trạng thái | Theo đúng định nghĩa | `O(C(2n - 1, n) * n)` | `O(C(2n - 1, n))` | Oracle, `n <= 9` |
| 2. Duyệt lại cây con | `|xu - nút|` từng cạnh | `O(n^2)` | `O(n)` | Dễ hiểu, `n <= 100` |
| 3. DFS hậu tự | Trả về thặng dư có dấu | `O(n)` | `O(h)` | **Cách nộp chính** |
| 4. BFS ngược | Dồn thặng dư lên cha | `O(n)` | `O(n)` | Tránh đệ quy |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(count_nodes(build_tree([1, 2, None, 3])))
print(tree_to_list(build_tree([0, None, 2])))
print(Solution().distributeCoins(build_tree([0, 0, 3])))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None
my_guess_2 = None
my_guess_3 = None

### Bài 2 — Tính tay (dễ-vừa)

Cây `[0, 5, 0, 0, 0]` (gốc `0`; con trái `5`, con phải `0`; con trái của nút `5` là `0` và con phải của nút `5` là `0`). Tính số lượt ít nhất bằng cách tính thặng dư từng cạnh.

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DFS hậu tự.

In [30]:
def exercise_3(root: Optional[TreeNode]) -> int:
    moves = 0

    def dfs(node: Optional[TreeNode]) -> int:
        nonlocal moves
        # TODO 1: node là None thì trả về 0
        # TODO 2: left, right = thặng dư có dấu của hai con
        # TODO 3: cộng |left| + |right| vào moves
        # TODO 4: trả về thặng dư có dấu của cây con gốc node
        pass

    dfs(root)
    return moves


# for vals, expected in examples:
#     assert exercise_3(build_tree(vals)) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(root)` bằng **BFS rồi xử lý ngược** (không đệ quy).

In [31]:
def exercise_4(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# for vals, expected in examples:
#     assert exercise_4(build_tree(vals)) == expected

### Bài 5 — Mini challenge: cạnh bận nhất (khó)

Viết `exercise_5(root)` trả về **số xu lớn nhất** phải đi qua **một cạnh** (tức `max |thặng dư|` trên mọi cạnh; cây một nút trả `0`).

```python
exercise_5(build_tree([3, 0, 0]))   -> 1
exercise_5(build_tree([0, 3, 0]))   -> 2
exercise_5(build_tree([0, 0, 3]))   -> 2
exercise_5(build_tree([1]))         -> 0
```
Gợi ý: giống Cách 3 nhưng thay phép cộng bằng `max`.

In [32]:
def exercise_5(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_5(build_tree([3, 0, 0])) == 1
# assert exercise_5(build_tree([0, 3, 0])) == 2
# assert exercise_5(build_tree([0, 0, 3])) == 2
# assert exercise_5(build_tree([1])) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `build_tree([1, 2, None, 3])`: gốc `1`, con trái `2`, con phải rỗng, con trái của `2` là `3`.
- Mức 2: cây có 3 nút thật; `[0, None, 2]` xuất lại đúng như vậy.
- Mức 3: `3`, `[0, None, 2]`, và `3` (con trái thiếu 1, con phải dư 2).

**Bài 2**
- Mức 1: hai lá con của nút `5` đều có `0` xu: thặng dư `-1` mỗi lá.
- Mức 2: nút `5`: `5 - 1 + (-1) + (-1) = 2`; con phải của gốc: `-1`.
- Mức 3: tổng `|-1| + |-1| + |2| + |-1| = 5`.

**Bài 3**
- Mức 1: `if node is None: return 0`.
- Mức 2: `left, right = dfs(node.left), dfs(node.right)`.
- Mức 3: `moves += abs(left) + abs(right); return node.val - 1 + left + right`.

**Bài 4**
- Mức 1: BFS bằng danh sách `nodes` và `parent`.
- Mức 2: `excess = [nd.val - 1 for nd in nodes]`.
- Mức 3: `for v in range(n - 1, 0, -1): ans += abs(excess[v]); excess[parent[v]] += excess[v]`.

**Bài 5**
- Mức 1: thặng dư có dấu vẫn tính như Cách 3.
- Mức 2: thay `moves += ...` bằng `best = max(best, abs(left), abs(right))`.
- Mức 3: trả về `best`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(count_nodes(build_tree([1, 2, None, 3])))
print(tree_to_list(build_tree([0, None, 2])))
print(Solution().distributeCoins(build_tree([0, 0, 3])))

assert count_nodes(build_tree([1, 2, None, 3])) == 3
assert tree_to_list(build_tree([0, None, 2])) == [0, None, 2]
assert Solution().distributeCoins(build_tree([0, 0, 3])) == 3

3
[0, None, 2]
3


### Đáp án Bài 2

Hai lá con của nút `5`: thặng dư `-1` và `-1` (cần 1 xu mỗi lá, mỗi cạnh 1 lượt). Nút `5`: `5 - 1 - 1 - 1 = 2`: cạnh lên gốc tốn 2 lượt. Con phải của gốc: `-1`: 1 lượt. Tổng `1 + 1 + 2 + 1 = 5`.

In [34]:
hand_answer_solution = 5
assert Solution().distributeCoins(build_tree([0, 5, 0, 0, 0])) == hand_answer_solution
assert moves_bfs(build_tree([0, 5, 0, 0, 0])) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(root: Optional[TreeNode]) -> int:
    moves = 0

    def dfs(node: Optional[TreeNode]) -> int:
        nonlocal moves
        if node is None:
            return 0
        left, right = dfs(node.left), dfs(node.right)
        moves += abs(left) + abs(right)
        return node.val - 1 + left + right

    dfs(root)
    return moves

for vals, expected in examples:
    assert exercise_3_solution(build_tree(vals)) == expected
rng = random.Random(3)
for _ in range(200):
    t = random_tree(rng, rng.randint(1, 60))
    assert exercise_3_solution(t) == sol.distributeCoins(t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(root: Optional[TreeNode]) -> int:
    nodes, parent = [root], [-1]
    i = 0
    while i < len(nodes):
        nd = nodes[i]
        for ch in (nd.left, nd.right):
            if ch is not None:
                nodes.append(ch)
                parent.append(i)
        i += 1
    excess = [nd.val - 1 for nd in nodes]
    ans = 0
    for v in range(len(nodes) - 1, 0, -1):
        ans += abs(excess[v])
        excess[parent[v]] += excess[v]
    return ans

for vals, expected in examples:
    assert exercise_4_solution(build_tree(vals)) == expected
rng = random.Random(4)
for _ in range(200):
    t = random_tree(rng, rng.randint(1, 60))
    assert exercise_4_solution(t) == sol.distributeCoins(t)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(root: Optional[TreeNode]) -> int:
    best = 0

    def dfs(node: Optional[TreeNode]) -> int:
        nonlocal best
        if node is None:
            return 0
        left, right = dfs(node.left), dfs(node.right)
        best = max(best, abs(left), abs(right))
        return node.val - 1 + left + right

    dfs(root)
    return best

assert exercise_5_solution(build_tree([3, 0, 0])) == 1
assert exercise_5_solution(build_tree([0, 3, 0])) == 2
assert exercise_5_solution(build_tree([0, 0, 3])) == 2
assert exercise_5_solution(build_tree([1])) == 0
rng = random.Random(5)
for _ in range(200):
    t = random_tree(rng, rng.randint(1, 40))
    nodes, parent = index_tree(t)
    s = [nd.val - 1 for nd in nodes]
    for v in range(len(nodes) - 1, 0, -1):
        s[parent[v]] += s[v]
    assert exercise_5_solution(t) == max([abs(x) for x in s[1:]] or [0])
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DFS hậu tự trả về thặng dư)

```python
class Solution:
    def distributeCoins(self, root: Optional[TreeNode]) -> int:
        moves = 0

        def dfs(node: Optional[TreeNode]) -> int:
            nonlocal moves
            if node is None:
                return 0
            left, right = dfs(node.left), dfs(node.right)
            moves += abs(left) + abs(right)          # luồng xu qua hai cạnh xuống con
            return node.val - 1 + left + right      # thặng dư CÓ DẤU của cây con

        dfs(root)
        return moves
```

Công thức nhớ nhanh: đáp án `= Σ_{cạnh} |xu trong cây con - số nút trong cây con|`.

### Checklist trước khi submit

- [ ] Hàm đệ quy trả về thặng dư **có dấu** (không phải `abs`)?
- [ ] Cộng `abs(left) + abs(right)` vào đáp án (mỗi cạnh đúng một lần)?
- [ ] Có trừ `1` cho chính nút (`node.val - 1 + ...`)?
- [ ] `dfs(None)` trả `0`?
- [ ] Dùng `nonlocal` (hoặc danh sách một phần tử) để cập nhật `moves` trong hàm lồng?
- [ ] Đã thử: một nút, cây dạng dây (`n(n-1)/2`), mọi xu ở một lá, ví dụ 1-2, `n = 100`?